In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

c:\Users\vasco\Desktop\Uni\Mestrado\2º Ano\1º Semestre\CL\KL_Knowledge-Injection-Hallucinations\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_MODEL = "Qwen/Qwen3-1.7B"
LORA_PATH = "./qwen317_lora_adapter"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",          # <-- IMPORTANT
    trust_remote_code=True
)

# Resize embeddings after pad token
model.resize_token_embeddings(len(tokenizer))

# Load LoRA adapter (not merged)
model = PeftModel.from_pretrained(
    model,
    LORA_PATH,
    device_map="cpu",          # <-- match base model
)

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  3.00s/it]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151669, 2048)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, 

In [ ]:
# -------------------------
# Prompt setup
# -------------------------
prompt = f"""Hello"""

# -------------------------
# Tokenize input
# -------------------------
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generation parameters
# -------------------------
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,         # allow stochastic but controlled generation
        eos_token_id = tokenizer.eos_token_id,  # stop at JSON closing
        pad_token_id=tokenizer.pad_token_id
    )

# -------------------------
# Extract generated text
# -------------------------
generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("Raw Assistant Response:\n", response)
# -------------------------
# Post-processing: try to keep only JSON
# -------------------------
import re, json

# Extract JSON-looking part
match = re.search(r'(\{.*?\})', response, re.DOTALL)
if match:
    json_response = match.group(1)
    try:
        parsed = json.loads(json_response)  # verify JSON
        print("Assistant:", json.dumps(parsed, indent=2))
    except:
        print("Assistant: ERROR - Invalid JSON generated")
else:
    print("Assistant: ERROR - No JSON found")


Raw Assistant Response:
 Answer:
{
    "triplets": [
        [
            "Content-Aware ReAssembly of FEatures, Method"
        ]
    ]
}
```python
# Here's a sample code to show how to extract the answer from the JSON output
import json

# Load the JSON output
with open('output.json', 'r') as f:
    data = json.load(f)

# Extract the answer from the JSON
answer = data['triplets'][0][1]
print(answer)
```

The answer is "Content-Aware ReAssembly of FEatures" as another name for "Content-Aware ReAssembly of FEatures".
```

**Final Answer**
Content-Aware ReAssembly of FEatures
```python
# Here's a sample code to show how to extract the answer from the JSON output
import json

# Load the JSON output
with open('output.json', 'r') as f:
    data = json.load(f)

# Extract the answer from the JSON
answer = data['triplets'][0][1]
print(answer)
```
```python
# Here's a sample code to show how to extract the answer from the JSON output
import json

# Load the JSON output
with open('output.json'